# Step 3 — High-risk recall and SHAP explainability
Input: `merged_dataset.csv`, `xgb_pipeline.pkl` and `model_meta.json` from Step 2.
Output: a tuned High-risk threshold in `model_meta.json`, plus SHAP plots in `/reports` for your write-up.

In [ ]:
# Cell 0 - environment and paths (works in Colab and locally)
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/incident-risk-predictor-files'
    !pip install -q xgboost imbalanced-learn shap
else:
    # Drive for desktop path; override with the IRP_BASE environment variable if yours differs
    BASE = os.environ.get('IRP_BASE', r'G:\My Drive\incident-risk-predictor-files')

for sub in ['datasets/raw', 'datasets/processed', 'model_checkpoints', 'reports', 'logs']:
    os.makedirs(os.path.join(BASE, sub), exist_ok=True)
print('IN_COLAB =', IN_COLAB, '| BASE =', BASE)

In [ ]:
# Cell 1 — Imports, reload the model and rebuild the same train/test split

import pandas as pd, numpy as np, joblib, json, shap
import matplotlib.pyplot as plt, seaborn as sns
from scipy import sparse
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

pipeline = joblib.load(f'{BASE}/model_checkpoints/xgb_pipeline.pkl')
meta = json.load(open(f'{BASE}/model_checkpoints/model_meta.json'))
FEATURES, LABELS = meta['features'], meta['labels']
RISK_MAP = {l: i for i, l in enumerate(LABELS)}
HIGH = RISK_MAP['High']

df = pd.read_csv(f'{BASE}/datasets/processed/merged_dataset.csv')
X, y = df[FEATURES], df['risk_level'].map(RISK_MAP)
X_train, X_test, y_train, y_test = train_test_split(   # same split as Step 2
    X, y, test_size=0.2, stratify=y, random_state=42)
print('Reloaded model ✓  Test set:', X_test.shape)

## Part A — Catch more High-risk incidents
High recall was 0.742, so about 1 in 4 High-risk incidents was labelled lower. In a maintenance setting that is the costly mistake.
The fix: predict **High** whenever P(High) passes a threshold *t*, instead of only when it is the largest probability.
We choose *t* on out-of-fold predictions from the **training** set, so the test set stays untouched for the final score.

In [ ]:
# Cell 2 — Out-of-fold probabilities on the training set (refits the pipeline 5 times)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_proba = cross_val_predict(pipeline, X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1)

def apply_threshold(proba, t):
    pred = proba.argmax(axis=1)
    pred[proba[:, HIGH] >= t] = HIGH
    return pred

rows = []
for t in np.arange(0.10, 0.51, 0.025):
    p = apply_threshold(oof_proba, t)
    rows.append({'threshold': round(t, 3),
                 'high_recall': recall_score(y_train, p, labels=[HIGH], average='macro'),
                 'high_precision': (p[y_train.values == HIGH] == HIGH).sum() / max((p == HIGH).sum(), 1),
                 'macro_f1': f1_score(y_train, p, average='macro')})
sweep = pd.DataFrame(rows).round(3)
base_f1 = f1_score(y_train, oof_proba.argmax(axis=1), average='macro')
print(f'Default (argmax) macro F1 on train OOF: {base_f1:.3f}')
print(sweep.to_string(index=False))

In [ ]:
# Cell 3 — Pick the threshold: best High recall while macro F1 stays within 0.01 of the default
ok = sweep[sweep['macro_f1'] >= base_f1 - 0.01]
best = ok.sort_values(['high_recall', 'macro_f1'], ascending=False).iloc[0]
HIGH_T = float(best['threshold'])
print(f'Chosen High threshold: {HIGH_T}')
print(best)

plt.figure(figsize=(7, 4))
plt.plot(sweep['threshold'], sweep['high_recall'], marker='o', label='High recall')
plt.plot(sweep['threshold'], sweep['macro_f1'], marker='o', label='Macro F1')
plt.axvline(HIGH_T, color='grey', ls='--', label=f'chosen t = {HIGH_T}')
plt.xlabel('P(High) threshold'); plt.legend(); plt.title('High-risk threshold sweep (train, out-of-fold)')
plt.tight_layout(); plt.savefig(f'{BASE}/reports/threshold_sweep.png', dpi=150); plt.show()

In [ ]:
# Cell 4 — Final score on the untouched test set: default vs tuned
test_proba = pipeline.predict_proba(X_test)
pred_default = test_proba.argmax(axis=1)
pred_tuned = apply_threshold(test_proba, HIGH_T)

for name, p in [('Default (argmax)', pred_default), (f'Tuned (t={HIGH_T})', pred_tuned)]:
    print(f'===== {name} =====')
    print(classification_report(y_test, p, target_names=LABELS, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, p) in zip(axes, [('Default', pred_default), (f'Tuned t={HIGH_T}', pred_tuned)]):
    sns.heatmap(confusion_matrix(y_test, p), annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig(f'{BASE}/reports/confusion_default_vs_tuned.png', dpi=150); plt.show()

meta['high_threshold'] = HIGH_T
json.dump(meta, open(f'{BASE}/model_checkpoints/model_meta.json', 'w'), indent=2)
print('✅ high_threshold saved to model_meta.json')

## Part B — SHAP: why the model predicts what it predicts

In [ ]:
# Cell 5 — Compute SHAP values on the test set
prep, model = pipeline.named_steps['prep'], pipeline.named_steps['model']
X_test_t = prep.transform(X_test)
if sparse.issparse(X_test_t):
    X_test_t = X_test_t.toarray()
feat_names = [n.split('__', 1)[1] for n in prep.get_feature_names_out()]

explainer = shap.TreeExplainer(model)
sv = np.asarray(explainer.shap_values(X_test_t))
if sv.shape[0] == len(LABELS) and sv.shape[1] == len(X_test_t):   # older SHAP: (classes, rows, features)
    sv = sv.transpose(1, 2, 0)
print('SHAP values shape (rows, features, classes):', sv.shape)

In [ ]:
# Cell 6 — Global importance, with one-hot columns grouped back to their original feature
def original_feature(name):
    for f in FEATURES:
        if name == f or name.startswith(f + '_'):
            return f
    return name

mean_abs = pd.DataFrame(np.abs(sv).mean(axis=0), index=feat_names, columns=LABELS)
grouped = mean_abs.groupby(original_feature).sum().sort_values('High')
grouped.plot(kind='barh', stacked=True, figsize=(8, 5),
             color={'Low': '#217346', 'Medium': '#FF8C00', 'High': '#C00000'})
plt.xlabel('mean |SHAP value|'); plt.title('Which inputs drive each risk level')
plt.tight_layout(); plt.savefig(f'{BASE}/reports/shap_global_importance.png', dpi=150); plt.show()
print(grouped.round(3).sort_values('High', ascending=False))

In [ ]:
# Cell 7 — Beeswarm for the High class: direction of each feature's effect
# (numeric values are on the scaled axis; red = high value, blue = low value)
shap.summary_plot(sv[:, :, HIGH], X_test_t, feature_names=feat_names, max_display=12, show=False)
plt.title('SHAP — what pushes an incident towards High risk')
plt.tight_layout(); plt.savefig(f'{BASE}/reports/shap_beeswarm_high.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Cell 8 — Explain one High-risk test incident (waterfall)
i = int(np.where(pred_tuned == HIGH)[0][0])
print('Incident inputs:'); print(X_test.iloc[i].to_string())
exp = shap.Explanation(values=sv[i, :, HIGH], base_values=explainer.expected_value[HIGH],
                       data=X_test_t[i], feature_names=feat_names)
shap.plots.waterfall(exp, max_display=10, show=False)
plt.title('Why this incident was predicted High risk')
plt.tight_layout(); plt.savefig(f'{BASE}/reports/shap_waterfall_example.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Cell 9 — The function Flask will use: prediction + top 3 reasons in plain words
def predict_with_reasons(incident: dict, top_n=3):
    row = pd.DataFrame([incident])[FEATURES]
    proba = pipeline.predict_proba(row)
    k = int(apply_threshold(proba, meta['high_threshold'])[0])
    xt = prep.transform(row)
    xt = xt.toarray() if sparse.issparse(xt) else xt
    s = np.asarray(explainer.shap_values(xt))
    s = s.transpose(1, 2, 0) if s.shape[0] == len(LABELS) and s.shape[1] == 1 else s
    contrib = pd.Series(s[0, :, k], index=feat_names).groupby(original_feature).sum()
    contrib = contrib.reindex(contrib.abs().sort_values(ascending=False).index)[:top_n]
    return {
        'risk_level': LABELS[k],
        'probabilities': {l: round(float(p), 3) for l, p in zip(LABELS, proba[0])},
        'top_factors': [{'feature': f, 'value': incident[f],
                         'effect': 'raises' if v > 0 else 'lowers', 'shap': round(float(v), 3)}
                        for f, v in contrib.items()],
    }

example = X_test.iloc[i].to_dict()
print(json.dumps(predict_with_reasons(example), indent=2, default=str))